## Data loading

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import ast
import pickle

In [ ]:
# CONSTANTS AND PATHS
TRAIN_DIR = "data/BBDD"
TEST_DIR = "data/qsd1_w1"

In [ ]:
def read_text(path):
    # CHANGED (only so it runs on a Mac): try UTF-8 first, then Latin-1 for accented names
    for enc in ("utf-8", "latin-1"):
        try:
            with open(path, "r", encoding=enc) as f:
                return f.read().strip()
        except UnicodeDecodeError:
            continue


def build_dataset(dataset_dir):
    """
    Build a dataset of images and their corresponding labels from a given directory.

    Parameters:
    dataset_dir (str): The path to the directory containing the dataset.

    Returns:
    pd.DataFrame: A DataFrame containing the image paths and their corresponding labels.
    """
    data = {}

    # CHANGED: sorted() so the row order is the same on every computer
    # (the ground truth refers to the images by position)
    for file in sorted(os.listdir(dataset_dir)):

        # CHANGED: skip hidden macOS files (.DS_Store, ._bbdd_00001.txt ...)
        if file.startswith("."):
            continue

        # Get filename without extension
        filename = os.path.splitext(file)[0]

        # Create entry if it doesn't already exist
        if filename not in data:
            data[filename] = {
                "image_path": None,
                "image": None,
                "artist": None,
                "artwork_name": None
            }

        # Load image
        if file.lower().endswith(".jpg"):
            image_path = os.path.join(dataset_dir, file)

            data[filename]["image_path"] = image_path
            data[filename]["image"] = cv2.imread(image_path)

        # Load label
        elif file.lower().endswith(".txt"):
            txt_path = os.path.join(dataset_dir, file)

            content = read_text(txt_path)

            # Empty .txt file
            if not content:
                continue

            try:
                lines = list(ast.literal_eval(content))

                if len(lines) >= 2:
                    data[filename]["artist"] = lines[0]
                    data[filename]["artwork_name"] = lines[1]

            except (ValueError, SyntaxError):
                print(f"Warning: Could not parse label file: {txt_path}")

    # Convert dictionary to DataFrame
    df = pd.DataFrame.from_dict(data, orient="index")

    # Reset index
    df = df.reset_index(drop=True)

    return df

In [ ]:
dataset = build_dataset(TRAIN_DIR)

In [ ]:
dataset.head()

## Basic histogram construction and plotting

In [ ]:
BINS = 128

In [ ]:
valid_images = dataset["image"].apply(
    lambda img: isinstance(img, np.ndarray)
    and img.ndim == 3
    and img.shape[2] == 3
)

dataset = dataset.loc[valid_images].reset_index(drop=True)

# Gray-level (intensity) histogram: convert to gray and count the pixels in each bin
dataset["histogram"] = dataset["image"].apply(
    lambda img: np.histogram(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), bins=BINS, range=(0, 256))[0]
)

In [ ]:
dataset.head()

In [ ]:
# Plot the gray histogram of one image
histogram = np.asarray(dataset["histogram"].iloc[12])

plt.figure(figsize=(10, 5))
plt.bar(np.arange(BINS), histogram, color="gray", width=1.0)
plt.title("Gray-level histogram of image 12")
plt.xlabel("Pixel Intensity (bin)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.title("Raw image 12")
plt.imshow(cv2.cvtColor(dataset["image"].iloc[12], cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## Similarity measure function

In [ ]:
def find_similar_images(target_image, dataset, top_n=5, method=cv2.HISTCMP_CORREL, plot=False, bins=256):
    """
    Find the most similar images in the dataset based on histogram comparison.

    Parameters:
    target_image (np.ndarray): The target image.
    dataset (pd.DataFrame): The dataset containing images and their histograms.
    top_n (int): The number of most similar images to return.

    Returns:
    pd.DataFrame: A DataFrame containing the top N most similar images and their similarity scores.
    """
    # Gray histogram of the target image
    target_histogram = np.histogram(cv2.cvtColor(target_image, cv2.COLOR_BGR2GRAY), bins=bins, range=(0, 256))[0]

    # CHANGED: use the requested comparison method (before it was always cv2.HISTCMP_CORREL)
    dataset["similarity"] = dataset["histogram"].apply(
        lambda hist: cv2.compareHist(target_histogram.astype(np.float32), hist.astype(np.float32), method)
    )

    # CHANGED: correlation and intersection are similarities (higher = more similar),
    # chi-squared and Bhattacharyya are distances (lower = more similar)
    higher_is_better = method in (cv2.HISTCMP_CORREL, cv2.HISTCMP_INTERSECT)
    similar_images = dataset.sort_values(by="similarity", ascending=not higher_is_better).head(top_n)

    if plot:
        plt.figure(figsize=(15, 5))
        plt.subplot(1, top_n + 1, 1)
        plt.imshow(cv2.cvtColor(target_image, cv2.COLOR_BGR2RGB))
        plt.title("Target Image")
        plt.axis("off")

        for i, (_, row) in enumerate(similar_images.iterrows(), start=2):
            plt.subplot(1, top_n + 1, i)
            plt.imshow(cv2.cvtColor(row["image"], cv2.COLOR_BGR2RGB))
            plt.title(f"Sim: {row['similarity']:.4f}")
            plt.axis("off")

        plt.tight_layout()
        plt.show()

    return similar_images[["image_path", "artist", "artwork_name", "similarity"]]

In [ ]:
test_dataset = build_dataset(TEST_DIR)

In [ ]:
test_dataset.head()

In [ ]:
find_similar_images(test_dataset["image"].iloc[3], dataset, top_n=5, plot=True, bins=BINS)

## Evaluation of bin number and distance metric

In [ ]:
import pickle

# Load the ground-truth correspondence for each test image.
correspondence_path = os.path.join(TEST_DIR, "gt_corresps.pkl")
with open(correspondence_path, "rb") as file:
    correspondences = pickle.load(file)

# Each entry contains the zero-based index of the matching training image.
correspondence_indices = [
    match[0] if match else None
    for match in correspondences
]

train_rows = dataset.reset_index(drop=True)
test_rows = test_dataset[test_dataset["image_path"].notna()].reset_index(drop=True)

correspondence_table = pd.DataFrame({
    "test_image": test_rows["image_path"],
    "train_index": correspondence_indices,
})
correspondence_table["train_image"] = correspondence_table["train_index"].apply(
    lambda index: train_rows.iloc[int(index)]["image_path"]
    if index is not None and 0 <= index < len(train_rows)
    else None
)
correspondence_table["artist"] = correspondence_table["train_index"].apply(
    lambda index: train_rows.iloc[int(index)]["artist"]
    if index is not None and 0 <= index < len(train_rows)
    else None
)
correspondence_table["artwork_name"] = correspondence_table["train_index"].apply(
    lambda index: train_rows.iloc[int(index)]["artwork_name"]
    if index is not None and 0 <= index < len(train_rows)
    else None
)

correspondence_table

In [ ]:
def create_bin_histograms(dataset, bins=BINS):
    """
    Create histograms for each image in the dataset.

    Parameters:
    dataset (pd.DataFrame): The dataset containing images.
    bins (int): The number of bins for the histogram.

    Returns:
    pd.DataFrame: The dataset with an additional column for histograms.
    """
    dataset["histogram"] = dataset["image"].apply(
        lambda img: np.histogram(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), bins=bins, range=(0, 256))[0]
    )
    return dataset

In [ ]:
binlist = [32, 64, 128, 256]

distance_metrics = {"Correlation":cv2.HISTCMP_CORREL,
                    "Intersect":cv2.HISTCMP_INTERSECT,
                    "Chi squared":cv2.HISTCMP_CHISQR,
                    "Battacharyya":cv2.HISTCMP_BHATTACHARYYA}

results = pd.DataFrame(columns=["bins", "distance_metric", "top1_accuracy", "top5_accuracy"])

# comparison using top 1 and top 5 accuracy with different histogram bin sizes
for bins in binlist:
    for distance_metric in distance_metrics.keys():
        print(f"Evaluating with {bins} bins and distance metric {distance_metric}:")
        dataset_with_histograms = create_bin_histograms(dataset.copy(), bins=bins)

        top1_correct = 0
        top5_correct = 0
        total_images = len(test_rows)

        for idx, test_row in test_rows.iterrows():
            target_image = test_row["image"]
            similar_images = find_similar_images(target_image, dataset_with_histograms, top_n=5, plot=False, bins=bins, method=distance_metrics[distance_metric])

            # Check if the correct match is in the top 1
            if similar_images.iloc[0]["image_path"] == correspondence_table.iloc[idx]["train_image"]:
                top1_correct += 1

            # Check if the correct match is in the top 5
            if correspondence_table.iloc[idx]["train_image"] in similar_images["image_path"].values:
                top5_correct += 1

        top1_accuracy = top1_correct / total_images
        top5_accuracy = top5_correct / total_images

        results.loc[len(results)] = {
            "bins": bins,
            "distance_metric": distance_metric,
            "top1_accuracy": top1_accuracy,
            "top5_accuracy": top5_accuracy
        }

        print(f"Top-1 Accuracy: {top1_accuracy:.4f}")
        print(f"Top-5 Accuracy: {top5_accuracy:.4f}\n")

In [ ]:
# Plot top-1 and top-5 accuracy for every distance metric and bin count
plot_data = results.melt(
    id_vars=["bins", "distance_metric"],
    value_vars=["top1_accuracy", "top5_accuracy"],
    var_name="metric",
    value_name="accuracy"
)
plot_data["metric"] = plot_data["metric"].map({
    "top1_accuracy": "Top-1 accuracy",
    "top5_accuracy": "Top-5 accuracy"
})

plt.figure(figsize=(14, 7))
sns.lineplot(
    data=plot_data,
    x="bins",
    y="accuracy",
    hue="distance_metric",
    style="metric",
    markers=True,
    dashes=True,
    linewidth=2.5,
    markersize=8
)

plt.title("Retrieval accuracy by histogram bins and distance metric")
plt.xlabel("Number of histogram bins")
plt.ylabel("Accuracy")
plt.xticks(binlist)
plt.ylim(0, 1)
plt.grid(axis="y", linestyle="--", alpha=0.35)
plt.legend(title="Distance metric / accuracy type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()